# People Analytics — Turnover de Colaboradores
## Notebook 2: Por que os colaboradores saem?

**Objetivo:** Identificar os fatores com maior correlação com o turnover — satisfação, horas extras, equilíbrio trabalho-vida, promoções e salário.

---

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/ibm_hr_attrition.csv')
df['saiu'] = (df['Attrition'] == 'Yes').astype(int)
print(f'{len(df)} colaboradores carregados.')

1470 colaboradores carregados.


## 1. Horas Extras — O fator mais visível

In [2]:
# Taxa de turnover por horas extras
overtime = df.groupby('OverTime')['saiu'].mean().reset_index()
overtime.columns = ['horas_extras', 'taxa']
overtime['taxa'] = overtime['taxa'] * 100
overtime['horas_extras'] = overtime['horas_extras'].map({'Yes': 'Com horas extras', 'No': 'Sem horas extras'})

fig = px.bar(
    overtime, x='horas_extras', y='taxa',
    color='horas_extras',
    color_discrete_map={'Com horas extras': '#E63946', 'Sem horas extras': '#457B9D'},
    title='Taxa de Turnover: Horas Extras fazem diferença?',
    labels={'horas_extras': '', 'taxa': 'Taxa de Turnover (%)'},
    text=overtime['taxa'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside', textfont_size=16)
fig.update_layout(showlegend=False, height=400)
fig.show()

com = overtime[overtime['horas_extras']=='Com horas extras']['taxa'].values[0]
sem = overtime[overtime['horas_extras']=='Sem horas extras']['taxa'].values[0]
print(f'Quem faz horas extras tem {com/sem:.1f}x mais chance de sair.')

Quem faz horas extras tem 2.9x mais chance de sair.


## 2. Satisfação no Trabalho e no Ambiente

In [3]:
# Turnover por nível de satisfação (trabalho e ambiente)
niveis = {1: '1 - Baixa', 2: '2 - Média', 3: '3 - Alta', 4: '4 - Muito Alta'}

sat_trabalho = df.groupby('JobSatisfaction')['saiu'].mean().reset_index()
sat_trabalho.columns = ['nivel', 'taxa']
sat_trabalho['taxa'] *= 100
sat_trabalho['nivel'] = sat_trabalho['nivel'].map(niveis)
sat_trabalho['tipo'] = 'Satisfação com o Trabalho'

sat_ambiente = df.groupby('EnvironmentSatisfaction')['saiu'].mean().reset_index()
sat_ambiente.columns = ['nivel', 'taxa']
sat_ambiente['taxa'] *= 100
sat_ambiente['nivel'] = sat_ambiente['nivel'].map(niveis)
sat_ambiente['tipo'] = 'Satisfação com o Ambiente'

sat_combined = pd.concat([sat_trabalho, sat_ambiente])

fig = px.line(
    sat_combined, x='nivel', y='taxa', color='tipo',
    markers=True,
    color_discrete_map={
        'Satisfação com o Trabalho': '#E63946',
        'Satisfação com o Ambiente': '#457B9D'
    },
    title='Quanto a satisfação influencia o turnover?',
    labels={'nivel': 'Nível de Satisfação', 'taxa': 'Taxa de Turnover (%)', 'tipo': ''}
)
fig.update_traces(line_width=3, marker_size=10)
fig.show()

## 3. Equilíbrio Trabalho-Vida

In [4]:
# WorkLifeBalance vs turnover
wlb_labels = {1: '1 - Ruim', 2: '2 - Regular', 3: '3 - Bom', 4: '4 - Excelente'}

wlb = df.groupby('WorkLifeBalance')['saiu'].mean().reset_index()
wlb.columns = ['nivel', 'taxa']
wlb['taxa'] *= 100
wlb['nivel'] = wlb['nivel'].map(wlb_labels)

fig = px.bar(
    wlb, x='nivel', y='taxa',
    color='taxa',
    color_continuous_scale=['#457B9D', '#E63946'],
    title='Equilíbrio Trabalho-Vida e Taxa de Turnover',
    labels={'nivel': 'Equilíbrio Trabalho-Vida', 'taxa': 'Taxa de Turnover (%)'},
    text=wlb['taxa'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 4. Promoções e Tempo sem Crescer

In [5]:
# Turnover por anos desde a última promoção
promocao = df.groupby('YearsSinceLastPromotion')['saiu'].agg(['mean', 'count']).reset_index()
promocao.columns = ['anos', 'taxa', 'total']
promocao['taxa'] *= 100
promocao = promocao[promocao['total'] >= 10]  # filtra grupos com poucos dados

fig = px.scatter(
    promocao, x='anos', y='taxa', size='total',
    color='taxa',
    color_continuous_scale=['#A8DADC', '#E63946'],
    title='Taxa de Turnover x Anos desde a Última Promoção',
    labels={
        'anos': 'Anos desde a Última Promoção',
        'taxa': 'Taxa de Turnover (%)',
        'total': 'Nº de Colaboradores'
    },
    trendline='ols'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 5. Viagens a Trabalho

In [6]:
# Business travel vs turnover
viagem = df.groupby('BusinessTravel')['saiu'].mean().reset_index()
viagem.columns = ['frequencia', 'taxa']
viagem['taxa'] *= 100
viagem = viagem.sort_values('taxa', ascending=False)

traducao = {
    'Travel_Frequently': 'Viaja com frequência',
    'Travel_Rarely': 'Viaja raramente',
    'Non-Travel': 'Não viaja'
}
viagem['frequencia'] = viagem['frequencia'].map(traducao)

fig = px.bar(
    viagem, x='frequencia', y='taxa',
    color='taxa',
    color_continuous_scale=['#A8DADC', '#E63946'],
    title='Frequência de Viagens e Taxa de Turnover',
    labels={'frequencia': '', 'taxa': 'Taxa de Turnover (%)'},
    text=viagem['taxa'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 6. Correlação Geral: Quais variáveis numéricas estão mais associadas ao turnover?

In [7]:
# Correlação de variáveis numéricas com turnover
colunas_constantes = [col for col in df.columns if df[col].nunique() == 1]
df_num = df.drop(columns=colunas_constantes + ['EmployeeNumber', 'Attrition'])

correlacoes = df_num.select_dtypes(include='number').corrwith(df_num['saiu'])
correlacoes = correlacoes.drop('saiu').sort_values()

cores = ['#E63946' if v > 0 else '#457B9D' for v in correlacoes.values]

fig = go.Figure(go.Bar(
    x=correlacoes.values,
    y=correlacoes.index,
    orientation='h',
    marker_color=cores
))
fig.update_layout(
    title='Correlação das Variáveis Numéricas com o Turnover',
    xaxis_title='Correlação de Pearson',
    yaxis_title='',
    height=600
)
fig.show()

print('\nTop 5 fatores que AUMENTAM o risco de saída:')
print(correlacoes.tail(5).sort_values(ascending=False).to_string())
print('\nTop 5 fatores que REDUZEM o risco de saída:')
print(correlacoes.head(5).to_string())


Top 5 fatores que AUMENTAM o risco de saída:
DistanceFromHome      0.077924
NumCompaniesWorked    0.043494
MonthlyRate           0.015170
PerformanceRating     0.002889
HourlyRate           -0.006846

Top 5 fatores que REDUZEM o risco de saída:
TotalWorkingYears    -0.171063
JobLevel             -0.169105
YearsInCurrentRole   -0.160545
MonthlyIncome        -0.159840
Age                  -0.159205


## 7. Resumo — Os 5 principais fatores de turnover

| # | Fator | Impacto | Interpretação |
|---|---|---|---|
| 1 | **Horas Extras** | Alto | Quem faz horas extras tem ~3x mais chance de sair |
| 2 | **Satisfação no Trabalho** | Alto | Nota 1 gera o dobro de turnover comparado à nota 4 |
| 3 | **Equilíbrio Trabalho-Vida** | Alto | Nota 1 (ruim) tem a maior taxa de saída |
| 4 | **Tempo sem Promoção** | Moderado | Quanto mais tempo sem crescer, maior a tendência de sair |
| 5 | **Viagens Frequentes** | Moderado | Quem viaja muito sai mais do que quem não viaja |

> **Conclusão:** Turnover não é sobre salário apenas. Os dados mostram que **condições de trabalho** — horas extras, satisfação e equilíbrio — explicam mais a saída do que a remuneração.
>
> **Próximo passo:** No Notebook 3, vamos construir um modelo preditivo para identificar quais colaboradores têm maior risco de sair — e quantificar o impacto financeiro do turnover.